<a href="https://colab.research.google.com/github/ishwariiic/AgriPredict/blob/main/Crop_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving District-wise crop production statistics data.csv to District-wise crop production statistics data.csv


In [2]:

!pip install -U pandas scikit-learn gradio


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 620.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.6/324.6 kB 22.2 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.11.1
    Uninstalling gradio_client-1.11.1:
      Successfully uninstalled gradio_client-1.11.1
  Attempting uninstall: gradio
    Found existing installation: gradio 5.42.0
    Uninstalli

In [3]:

import pandas as pd
import numpy as np
import gradio as gr
import warnings
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from google.colab import files

warnings.filterwarnings("ignore")

In [4]:
print("👉 Please upload your APY CSV (from DES portal)...")
uploaded = files.upload()
csv_path = list(uploaded.keys())[0]


def read_and_clean(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]


    def pick(colnames, candidates):
        s = {c.lower(): c for c in colnames}
        for cand in candidates:
            for c in colnames:
                if cand in c.lower():
                    return c
        return None

    cols = {}
    cols['state'] = pick(df.columns, ['state','state/ut'])
    cols['district'] = pick(df.columns, ['district'])
    cols['crop'] = pick(df.columns, ['crop'])
    cols['season'] = pick(df.columns, ['season'])
    cols['year'] = pick(df.columns, ['year'])
    cols['area'] = pick(df.columns, ['area'])
    cols['production'] = pick(df.columns, ['production'])
    cols['yield'] = pick(df.columns, ['yield'])

    if cols['district'] is None:
        df['district'] = 'Unknown'
        cols['district'] = 'district'

    if cols['yield'] is None:
        df['Yield (Kg/Ha) [computed]'] = (pd.to_numeric(df[cols['production']], errors='coerce') * 1000.0) / pd.to_numeric(df[cols['area']], errors='coerce')
        cols['yield'] = 'Yield (Kg/Ha) [computed]'

    nd = pd.DataFrame({
        'State': df[cols['state']].astype(str).str.title().str.strip(),
        'District': df[cols['district']].astype(str).str.title().str.strip(),
        'Crop': df[cols['crop']].astype(str).str.title().str.strip(),
        'Season': df[cols['season']].astype(str).str.title().str.strip(),
        'Year': pd.to_numeric(df[cols['year']], errors='coerce'),
        'Area_Ha': pd.to_numeric(df[cols['area']], errors='coerce'),
        'Production_Tonnes': pd.to_numeric(df[cols['production']], errors='coerce'),
        'Yield_KgPerHa': pd.to_numeric(df[cols['yield']], errors='coerce'),
    }).dropna()

    return nd[(nd['Area_Ha']>0) & (nd['Yield_KgPerHa']>0)]

df = read_and_clean(csv_path)
print("✅ Data loaded with shape:", df.shape)

👉 Please upload your APY CSV (from DES portal)...


Saving District-wise crop production statistics data.csv to District-wise crop production statistics data (1).csv
✅ Data loaded with shape: (18219, 8)


In [5]:
cat_features = ['State','District','Crop','Season']
num_features = ['Year','Area_Ha']

X = df[cat_features + num_features]
y = df['Yield_KgPerHa']

pre = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=10), cat_features),
    ('num', 'passthrough', num_features)
])

model = Pipeline([
    ('pre', pre),
    ('rf', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
])

model.fit(X,y)

print("✅ Model trained")

✅ Model trained


In [6]:

def predict_yield(state, crop, season, year, area, district):
    row = pd.DataFrame([{
        'State': state,
        'District': district if district else "Unknown",
        'Crop': crop,
        'Season': season,
        'Year': int(year),
        'Area_Ha': float(area) if area else 1.0
    }])
    yhat = float(model.predict(row)[0])
    production = (yhat * row['Area_Ha'].iloc[0]) / 1000.0
    return round(yhat,2), round(production,2)

def region_wise(crop, season, year):
    states = sorted(df['State'].unique().tolist())
    rows = []
    for st in states:
        yhat,_ = predict_yield(st, crop, season, year, 1000.0, "Unknown")
        rows.append([st, yhat])
    return pd.DataFrame(rows, columns=["State","Predicted_Yield (Kg/Ha)"])


states = sorted(df['State'].unique())
crops = sorted(df['Crop'].unique())
seasons = sorted(df['Season'].unique())
years = sorted(df['Year'].unique())

with gr.Blocks() as demo:
    gr.Markdown("# 🌾 Indian Crop Yield Predictor \nUpload APY dataset CSV → Train → Predict")

    with gr.Tab("Single Prediction"):
        state = gr.Dropdown(states, label="State")
        crop = gr.Dropdown(crops, label="Crop")
        season = gr.Dropdown(seasons, label="Season")
        year = gr.Slider(min(years), max(years), step=1, value=max(years), label="Year")
        district = gr.Textbox(label="District (optional)")
        area = gr.Number(label="Area (Hectare)", value=100.0)
        btn = gr.Button("Predict Yield")
        y_out = gr.Number(label="Predicted Yield (Kg/Ha)")
        p_out = gr.Number(label="Estimated Production (Tonnes)")
        btn.click(predict_yield, [state,crop,season,year,area,district],[y_out,p_out])

    with gr.Tab("Region-wise Yields"):
        crop2 = gr.Dropdown(crops, label="Crop")
        season2 = gr.Dropdown(seasons, label="Season")
        year2 = gr.Slider(min(years), max(years), step=1, value=max(years), label="Year")
        btn2 = gr.Button("Predict State-wise Yields")
        table = gr.Dataframe(headers=["State","Predicted_Yield (Kg/Ha)"], datatype=["str","number"])
        btn2.click(region_wise, [crop2,season2,year2], table)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f88f3095ba9cfc9187.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
